# conv-windowing-1d — worked example 1: Stride-2 1-D conv window view via as_strided

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-windowing-1d`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A stride-1 conv window view duplicates the input's `W`-axis stride onto both the new output-width (`OW`) axis and the kernel-width (`KW`) axis. For a **stride-`S`** convolution, the `OW` axis instead advances by `S` elements per output position, so its stride becomes `S * s_w` while the `KW` axis keeps `s_w`. The output width shrinks to `OW = (W - KW) // S + 1`.

## Worked solution

**Goal.** Build a `(B, IC, OW, KW)` view where adjacent windows are offset by `S` input elements, then confirm contracting it with a kernel matches `F.conv1d(..., stride=S)`.

1. **Unpack the shape and strides.** `B, IC, W = x.shape` and `s_b, s_ic, s_w = x.stride()`. The strides tell us how many storage elements to jump to move one step along each existing axis.
2. **Compute the output width.** With stride `S`, the last valid window starts at index `W - KW`, and starts are spaced `S` apart, so `OW = (W - KW) // S + 1`.
3. **Pick the new strides.** The `KW` axis walks one element at a time inside a window, so its stride is `s_w`. The `OW` axis jumps a whole stride of `S` input elements between windows, so its stride is `S * s_w`. The batch/channel strides are unchanged.
4. **Call as_strided.** `x.as_strided(size=(B, IC, OW, KW), stride=(s_b, s_ic, S * s_w, s_w))`. This is a pure view — no copy — so it shares storage with `x`.
5. **Why it equals conv1d.** Each window is a length-`KW` slice; einsum `'b i o k, c i k -> b c o'` contracts the kernel against every window, which is exactly the dot-products `F.conv1d` computes at stride `S`.

In [ ]:
import torch as t
import torch.nn.functional as F
from einops import einsum

def conv1d_windows_strided(x: t.Tensor, KW: int, S: int) -> t.Tensor:
    B, IC, W = x.shape
    OW = (W - KW) // S + 1
    s_b, s_ic, s_w = x.stride()
    return x.as_strided(size=(B, IC, OW, KW), stride=(s_b, s_ic, S * s_w, s_w))

t.manual_seed(0)
x = t.randn(2, 3, 11)
weight = t.randn(4, 3, 4)
KW, S = 4, 2
win = conv1d_windows_strided(x, KW, S)
out = einsum(win, weight, 'b i o k, c i k -> b c o')
ref = F.conv1d(x, weight, stride=S)
print('window shape:', tuple(win.shape))
print('shares storage:', win.data_ptr() == x.data_ptr())
print('matches conv1d:', t.allclose(out, ref, atol=1e-4))